In [4]:
import hashlib
import os
import platform
import uuid

# 修改为自己的学号
STUDENT_ID = 20244081117

CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV")
print(f"[Conda 环境] {CONDA_ENV}")

MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号]     {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID]   {RUN_ID}")

[Conda 环境] py_dateprocessing
[学号]     20244081117
[机器指纹] c8328ad6c13c29ec
[RUN_ID]   20244081117-c8328ad6c13c29ec


Python 正则表达式练习

In [5]:
import re

# 1. 检查字符串是否仅包含英文字母 a-z、A-Z 和数字 0-9
def check_alnum(s):
    return bool(re.fullmatch(r'[a-zA-Z0-9]+', s))

# 2. 匹配一个 a 后面跟着零个或多个 b 的字符串
def match_a_followed_by_bs(s):
    return re.fullmatch(r'ab*', s)

# 3. 匹配一个 a 后面跟着一个或多个 b 的字符串
def match_a_followed_by_one_or_more_bs(s):
    return re.fullmatch(r'ab+', s)

# 4. 匹配以 a 开头、以 b 结尾，且中间可以有任意字符的字符串
def match_a_start_b_end(s):
    return re.fullmatch(r'a.*b', s, re.DOTALL)

# 5. 删除 IP 地址中各段数字的前导零（全零段保留一个 0）
def strip_ip_leading_zeros(ip):
    return re.sub(r'\b0+\d*\b', lambda m: str(int(m.group(0))), ip)

# 6. 将空格替换为下划线，并将下划线替换为空格
def swap_spaces_underscores(s):
    p = '\x00'
    return s.replace(' ', p).replace('_', ' ').replace(p, '_')

# 7. 将 yyyy-mm-dd 格式的日期转换为 dd-mm-yyyy 格式
def reformat_date(date_str):
    return re.sub(r'(\d{4})-(\d{2})-(\d{2})', r'\3-\2-\1', date_str)

# 8. 找出给定字符串中所有以 a 或 e 开头的单词
def find_words_starting_with_a_or_e(s):
    return re.findall(r'\b[ae]\w*', s, re.IGNORECASE)

# 9. 将给定字符串中的 Road 缩写为 Rd
def abbreviate_road(s):
    return re.sub(r'\bRoad\b', 'Rd', s, flags=re.IGNORECASE)

# 10. 找出字符串中所有长度至少为 4 个字符的单词
def find_long_words(s):
    return re.findall(r'\b\w{4,}\b', s)

# 11. 删除字符串中除字母和数字之外的所有字符
def remove_non_alnum(s):
    return re.sub(r'[^a-zA-Z0-9]', '', s)

# 12. 删除字符串中的括号及括号内的内容
def remove_parentheses_content(s):
    return re.sub(r'\s*\([^)]*\)', '', s)


if __name__ == '__main__':
    print("1:", check_alnum("Hello123"), check_alnum("Hello 123"))
    print("2:", bool(match_a_followed_by_bs("a")), bool(match_a_followed_by_bs("abbb")), match_a_followed_by_bs("ac"))
    print("3:", bool(match_a_followed_by_one_or_more_bs("ab")), bool(match_a_followed_by_one_or_more_bs("abbb")), match_a_followed_by_one_or_more_bs("a"))
    print("4:", bool(match_a_start_b_end("aXYZb")), bool(match_a_start_b_end("ab")), match_a_start_b_end("aXbY"))
    print("5:", strip_ip_leading_zeros("192.168.001.010"), strip_ip_leading_zeros("010.000.001.100"))
    print("6:", swap_spaces_underscores("hello_world foo_bar"))
    print("7:", reformat_date("2024-03-15"))
    print("8:", find_words_starting_with_a_or_e("An apple a day keeps everyone eager"))
    print("9:", abbreviate_road("Main Road and Side road"))
    print("10:", find_long_words("This is a test of long words"))
    print("11:", remove_non_alnum("Hello, World! 123"))
    print("12:", [remove_parentheses_content(x) for x in ["example (.com)", "w3resource", "github (.com)", "stackoverflow (.com)"]])

1: True False
2: True True None
3: True True None
4: True True None
5: 192.168.1.10 10.0.1.100
6: hello world_foo bar
7: 15-03-2024
8: ['An', 'apple', 'a', 'everyone', 'eager']
9: Main Rd and Side Rd
10: ['This', 'test', 'long', 'words']
11: HelloWorld123
12: ['example', 'w3resource', 'github', 'stackoverflow']


从网页检索气候数据

In [8]:
import requests
from bs4 import BeautifulSoup


def _fetch_html(url):
    resp = requests.get(url, timeout=15, headers={'User-Agent': 'Mozilla/5.0'})
    resp.raise_for_status()
    return resp.text


# 1. 定位气候数据表格
def retrieve_climate_table(s):
    soup = BeautifulSoup(_fetch_html(s), 'html.parser')
    for table in soup.find_all('table'):
        if 'Climate data for' in ' '.join(table.get_text().split()):
            return table
    return None


# 2. 列出气候数据表格的行名
def list_climate_table_row_names(s):
    table = retrieve_climate_table(s)
    if table is None:
        return None
    names = []
    for tr in table.find_all('tr'):
        th = tr.find('th')
        if not th:
            continue
        name = ' '.join(th.get_text().split())
        low = name.lower()
        if (not name
                or 'climate data for' in low
                or low.startswith('source')
                or name == 'Month'):
            continue
        names.append(name)
    return names


# 3. 用多个城市页面测试，包括结构差异页面
if __name__ == '__main__':
    urls = [
        "https://en.wikipedia.org/wiki/Madison,_Wisconsin",
        "https://en.wikipedia.org/wiki/Buenos_Aires",
        "https://en.wikipedia.org/wiki/Harbin",
        "https://en.wikipedia.org/wiki/Los_Angeles",
        "https://en.wikipedia.org/wiki/Hyderabad",
        "https://en.wikipedia.org/wiki/Boston",
    ]
    for url in urls:
        print(f"\n=== {url} ===")
        try:
            table = retrieve_climate_table(url)
            if table is None:
                print("未找到气候数据表格")
            else:
                children = [c.name for c in table.children if getattr(c, 'name', None)]
                print(f"直接子节点数: {len(list(table.children))}, 标签示例: {children[:5]}")
                rows = list_climate_table_row_names(url)
                print(f"行名数量: {len(rows)}, 前10个: {rows[:10]}")
        except Exception as e:
            print(f"错误: {e}")


=== https://en.wikipedia.org/wiki/Madison,_Wisconsin ===
直接子节点数: 2, 标签示例: ['tbody']
行名数量: 15, 前10个: ['Record high °F (°C)', 'Mean maximum °F (°C)', 'Mean daily maximum °F (°C)', 'Daily mean °F (°C)', 'Mean daily minimum °F (°C)', 'Mean minimum °F (°C)', 'Record low °F (°C)', 'Average precipitation inches (mm)', 'Average snowfall inches (cm)', 'Average extreme snow depth inches (cm)']

=== https://en.wikipedia.org/wiki/Buenos_Aires ===
直接子节点数: 2, 标签示例: ['tbody']
行名数量: 11, 前10个: ['Record high °C (°F)', 'Mean daily maximum °C (°F)', 'Daily mean °C (°F)', 'Mean daily minimum °C (°F)', 'Record low °C (°F)', 'Average precipitation mm (inches)', 'Average precipitation days (≥ 0.1 mm)', 'Average relative humidity (%)', 'Mean monthly sunshine hours', 'Mean daily sunshine hours']

=== https://en.wikipedia.org/wiki/Harbin ===
直接子节点数: 2, 标签示例: ['tbody']
行名数量: 14, 前10个: ['Record high °C (°F)', 'Mean maximum °C (°F)', 'Mean daily maximum °C (°F)', 'Daily mean °C (°F)', 'Mean daily minimum °C (°F)',